# Richet Khola Small Hydropower Project, Nepal
## Computational Analysis of Hydraulic Performance and Energy Generation

**Research project | Python / Google Colab**

**Data integrity:** Project specifications below are *preliminary and require independent confirmation*. Monthly river discharges and penstock dimensions are **illustrative assumptions, not site measurements**. The uploaded 12 MW drawing is **not used** because its association with the 4.98 MW project is unconfirmed. This notebook is a computational case study, not a validated operational audit.

### Objectives
1. Calculate hydraulic power and implied overall efficiency.
2. Explore efficiency and discharge sensitivity.
3. Model illustrative monthly energy and annual capacity factor.
4. Evaluate assumed penstock friction and diameter effects.
5. Explore power response to discharge and efficiency, and rank local parameter sensitivities.

### Preliminary inputs
- Installed capacity: 4.98 MW
- Design discharge: 2.25 m³/s
- Rated net head: 258.15 m
- Annual design energy: 27.27 GWh

**Source to verify:** Richet Jalbidhyut Company Limited project information (https://richethydro.com/projects/). Confirm against official project design documents before publication.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

rho, g = 1000.0, 9.81
capacity_MW, design_Q, rated_net_head = 4.98, 2.25, 258.15
design_energy_GWh = 27.27
hydraulic_MW = rho*g*design_Q*rated_net_head/1e6
implied_eta = capacity_MW/hydraulic_MW
max_energy_GWh = capacity_MW*8760/1000
print(f"Theoretical hydraulic power: {hydraulic_MW:.4f} MW")
print(f"Implied overall efficiency: {100*implied_eta:.2f}%")
print(f"Design capacity factor: {100*design_energy_GWh/max_energy_GWh:.2f}%")
assert 0 < implied_eta < 1

## 1. Efficiency sensitivity
For constant net head and discharge, power is $P=\rho g QH\eta$. The uncapped calculation illustrates the physics; actual plant output is constrained by rated capacity.

In [ ]:
eff = np.arange(.60,.951,.05)
power_uncapped = hydraulic_MW*eff
fig, ax=plt.subplots(figsize=(8,4.5))
ax.plot(eff*100,power_uncapped,'o-',label='Uncapped calculation')
ax.axhline(capacity_MW,ls='--',color='tab:red',label='Installed capacity')
ax.set(xlabel='Overall efficiency (%)',ylabel='Electrical power (MW)',title='Efficiency sensitivity at design discharge')
ax.grid(alpha=.25);ax.legend();plt.show()
display(pd.DataFrame({'Efficiency (%)':eff*100,'Uncapped power (MW)':power_uncapped}).round(3))

## 2. Illustrative seasonal model
The monthly river discharge series below is **synthetic**. Available turbine discharge is capped at design flow; power is capped at rated capacity. The model assumes 24-hour availability, constant net head and constant overall efficiency. It omits environmental releases, outages, and part-load efficiency.

In [ ]:
months=['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
river_Q=np.array([.65,.55,.60,.85,1.30,1.90,2.60,2.85,2.40,1.75,1.10,.80])
days=np.array([31,28,31,30,31,30,31,31,30,31,30,31])
used_Q=np.minimum(river_Q,design_Q)
monthly_power=np.minimum(rho*g*used_Q*rated_net_head*implied_eta/1e6,capacity_MW)
monthly_GWh=monthly_power*24*days/1000
df=pd.DataFrame({'Month':months,'Illustrative river discharge (m3/s)':river_Q,'Turbine discharge (m3/s)':used_Q,'Power (MW)':monthly_power,'Energy (GWh)':monthly_GWh})
display(df.round(3))
fig,axs=plt.subplots(2,1,figsize=(9,7),layout='constrained')
axs[0].plot(months,river_Q,'o-');axs[0].axhline(design_Q,ls='--',color='tab:red',label='Design flow');axs[0].set_ylabel('River discharge (m³/s)');axs[0].legend();axs[0].grid(alpha=.25)
axs[1].bar(months,monthly_GWh);axs[1].set_ylabel('Modelled energy (GWh)');axs[1].set_xlabel('Month');axs[1].grid(axis='y',alpha=.25)
fig.suptitle('Illustrative seasonal hydropower scenario');plt.show()

## 3. Annual energy comparison
These are scenario outputs, **not actual Richet Khola generation measurements**.

In [ ]:
annual_GWh=monthly_GWh.sum()
cf=annual_GWh/max_energy_GWh*100
diff=annual_GWh-design_energy_GWh
summary=pd.DataFrame({'Metric':['Illustrative model annual energy (GWh)','Preliminary design annual energy (GWh)','Theoretical maximum annual energy (GWh)','Model capacity factor (%)','Design capacity factor (%)','Model minus design (GWh)','Difference relative to design (%)'], 'Value':[annual_GWh,design_energy_GWh,max_energy_GWh,cf,100*design_energy_GWh/max_energy_GWh,diff,100*diff/design_energy_GWh]})
display(summary.round(4))
fig,ax=plt.subplots(figsize=(8,4));ax.bar(['Illustrative model','Preliminary design','Theoretical maximum'],[annual_GWh,design_energy_GWh,max_energy_GWh]);ax.set_ylabel('Annual energy (GWh)');ax.set_title('Annual energy comparison');plt.show()

## 4. Illustrative penstock hydraulics
**Assumptions only:** length 1,200 m, internal diameter 1.20 m, constant Darcy friction factor 0.02. These are not dimensions extracted from project drawings.

Darcy–Weisbach: $h_f=f(L/D)V^2/(2g)$ and $V=4Q/(\pi D^2)$.

To avoid subtracting friction twice from an already **net** rated head, the scenario *infers* gross head by adding the assumed design-flow friction loss to the rated net head. This does not estimate actual gross head or minor losses.

In [ ]:
L,D,f=1200.,1.20,.02
area=np.pi*D**2/4

def friction_loss(Q,D,L=L,f=f):
    V=4*np.asarray(Q)/(np.pi*D**2)
    return f*(L/D)*V**2/(2*g)

hf_design=float(friction_loss(design_Q,D))
assumed_gross_head=rated_net_head+hf_design
V_design=design_Q/area
hydraulic_loss_MW=rho*g*design_Q*hf_design/1e6
print(f"Velocity: {V_design:.3f} m/s; friction head loss: {hf_design:.3f} m")
print(f"Assumed gross head: {assumed_gross_head:.3f} m; hydraulic friction power: {hydraulic_loss_MW:.4f} MW")
Ds=np.linspace(.8,2.,150)
hfs=friction_loss(design_Q,Ds)
fig,ax=plt.subplots(figsize=(8,4.5));ax.plot(Ds,hfs);ax.set(xlabel='Assumed pipe diameter (m)',ylabel='Friction head loss (m)',title='Diameter sensitivity: assumed 1,200 m penstock');ax.grid(alpha=.25);plt.show()

## 5. Power performance map
This scenario varies discharge and overall efficiency, with constant assumed gross head and friction factor. It applies the 4.98 MW capacity cap. Real turbine efficiency is generally flow-dependent.

In [ ]:
qvals=np.linspace(.1,design_Q,80)
etavals=np.linspace(.60,.95,70)
QQ,EE=np.meshgrid(qvals,etavals)
HH=np.maximum(assumed_gross_head-friction_loss(QQ,D),0)
P_uncapped=rho*g*QQ*HH*EE/1e6
P=np.minimum(P_uncapped,capacity_MW)
fig=plt.figure(figsize=(10,6));ax=fig.add_subplot(111,projection='3d')
surf=ax.plot_surface(QQ,EE*100,P,cmap='viridis',linewidth=0)
ax.set(xlabel='Discharge (m³/s)',ylabel='Efficiency (%)',zlabel='Power (MW)',title='Illustrative capped power surface')
fig.colorbar(surf,ax=ax,shrink=.6,label='Power (MW)');plt.show()
fig,ax=plt.subplots(figsize=(9,5));im=ax.imshow(P,origin='lower',aspect='auto',extent=[qvals.min(),qvals.max(),etavals.min()*100,etavals.max()*100],cmap='viridis');ax.set(xlabel='Discharge (m³/s)',ylabel='Overall efficiency (%)',title='Illustrative power heatmap');fig.colorbar(im,ax=ax,label='Power (MW)');plt.show()

## 6. Normalised sensitivity analysis
Each parameter is varied ±10%, holding the other parameters fixed. We use **uncapped** electrical power to avoid artificial zero sensitivities at the rated capacity ceiling. Gross head is fixed to the illustrative baseline when varying other parameters.

In [ ]:
def uncapped_power(Q,Hgross,eta,D):
    return rho*g*Q*max(Hgross-float(friction_loss(Q,D)),0)*eta/1e6
baseline={'Discharge':design_Q,'Gross head':assumed_gross_head,'Overall efficiency':implied_eta,'Penstock diameter':D}
def evaluate(d):return uncapped_power(d['Discharge'],d['Gross head'],d['Overall efficiency'],d['Penstock diameter'])
p0=evaluate(baseline)
rows=[]
for name,val in baseline.items():
    low=baseline.copy();high=baseline.copy();low[name]=val*.9;high[name]=val*1.1
    pl,ph=evaluate(low),evaluate(high)
    rows.append({'Parameter':name,'Power at -10% (MW)':pl,'Power at +10% (MW)':ph,'Normalised sensitivity':(ph-pl)/(.2*p0)})
sensitivity=pd.DataFrame(rows)
display(sensitivity.round(4))
fig,ax=plt.subplots(figsize=(8,4));ax.barh(sensitivity['Parameter'],sensitivity['Normalised sensitivity']);ax.set_xlabel('Normalised sensitivity coefficient');ax.set_title('One-at-a-time ±10% sensitivity');ax.grid(axis='x',alpha=.25);plt.show()

## 7. Interpretation, limitations and conclusion

**Interpretation.** Under the preliminary specifications, theoretical hydraulic power is approximately 5.70 MW and the rated electrical output implies an overall conversion efficiency near 87.4%. In the *illustrative* seasonal scenario, wet-season discharge supports higher generation and dry-season discharge reduces power. The diameter sensitivity illustrates the strong dependence of friction loss on pipe diameter; the sensitivity analysis quantifies local parameter effects under fixed assumptions.

**Limitations.** There are no verified monthly discharge observations, turbine hill charts, environmental-flow deductions, measured generation records, or confirmed penstock dimensions in this model. Gross head is an illustrative inferred quantity. The friction factor is held constant, minor losses and transients are omitted, and maintenance outages are not modelled. Apparent agreement with preliminary annual design energy **does not validate** the synthetic flows.

**Conclusion.** This reproducible notebook provides a physics-based framework for investigating run-of-river hydropower generation. It is suitable as a computational student project, provided that synthetic inputs are explicitly distinguished from field data. The next scientific improvement is to replace assumed inputs with authenticated Richet Khola design and operating records.

### References / verification checklist
- Richet Jalbidhyut Company Limited, project page: https://richethydro.com/projects/ — **verify** preliminary plant specifications.
- Standard fluid mechanics: Darcy–Weisbach head-loss relation and continuity equation.
- Project-specific hydrology, turbine efficiency, drawings and energy records: **not yet verified or incorporated**.

### Data export
The CSVs below contain **scenario-model outputs**, not site measurements.

In [ ]:
df.to_csv('Richet_Khola_Illustrative_Monthly_Model.csv',index=False)
summary.to_csv('Richet_Khola_Scenario_Summary.csv',index=False)
sensitivity.to_csv('Richet_Khola_Sensitivity.csv',index=False)
print('Exported 3 CSV files.')